# 🖋️ Remote GPU Model Training & Verification
### Signature Detection (RT-DETR-R50 / YOLOX-S) & Metric Verification (ResNet-50)
**License:** Apache 2.0 & BSD (Strictly Non-Ultralytics, Zero AGPL)

This notebook trains and benchmarks your models on Google Colab GPU (T4/A100) using your GitHub repository [sign-detect-verify](https://github.com/9m2usman/sign-detect-verify).

> **Note on API Keys:** No external API keys (Comet ML / MLflow) are required! Tracking is set to local mode by default.

## Step 1: Check GPU & Clone GitHub Repository

In [1]:
# Verify GPU allocation
!nvidia-smi

# Clone latest code from your GitHub repository
!rm -rf sign-detect-verify
!git clone https://github.com/9m2usman/sign-detect-verify.git
%cd sign-detect-verify

# Install dependencies (strictly Apache 2.0 / MIT / BSD)
!pip install -q -r requirements.txt

# Verify zero AGPL Ultralytics contamination
!python -c "import ultralytics" 2>/dev/null && echo "WARNING: AGPL Detected!" || echo "[✓] Clean Environment: No AGPL Ultralytics installed."

Sun Nov  8 12:16:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2: Download & Extract Dataset
Mount your Google Drive (if you uploaded `merged_signature_dataset.zip` there) OR use sample fixtures.

In [ ]:
# Option A: Mount Google Drive (if your dataset zip is on Drive)
import os
import zipfile

try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_zip = '/content/drive/MyDrive/merged_signature_dataset.zip'
    if os.path.exists(drive_zip):
        print("[*] Extracting dataset from Google Drive...")
        with zipfile.ZipFile(drive_zip, 'r') as zip_ref:
            zip_ref.extractall('.')
        print("[+] Dataset extracted successfully!")
    else:
        print("[*] Using local repository samples and downloaded weights.")
except Exception as e:
    print("[*] Proceeding without Drive mount:", e)

## Step 3: Train Detection Models (Stage A)
Train **RT-DETR-R50** (DEtection TRansformer) — uses local logging (`--tracker none`, NO API key needed).

In [ ]:
# Run RT-DETR-R50 training on GPU (No Comet API key required)
!python training/detection/train_rtdetr.py \
    --dataset-dir ./merged_signature_dataset \
    --batch-size 8 \
    --epochs 30 \
    --output-dir ./checkpoints/rtdetr_r50 \
    --tracker none

## Step 4: Train Verification Model (Stage B)
Train **ResNet50** metric feature extractor using Contrastive Loss (`--tracker none`, NO API key needed).

In [ ]:
!python training/verification/train_resnet50.py \
    --batch-size 32 \
    --epochs 25 \
    --output-dir ./checkpoints/resnet50_verifier \
    --tracker none

## Step 5: Test End-to-End Pipeline (Document + Reference Signature)
Run the complete detection $\to$ cropping $\to$ embedding $\to$ cosine similarity pipeline on GPU.

In [ ]:
# Test 1: Authentic Reference (Expected: VERIFIED)
!python verify_document.py \
    --doc sample_test_run/sample_document.jpg \
    --ref sample_test_run/reference_authentic.jpg

# Test 2: Impostor Reference (Expected: FAILED_VERIFICATION)
!python verify_document.py \
    --doc sample_test_run/sample_document.jpg \
    --ref sample_test_run/reference_impostor.jpg